In [6]:
from sqlalchemy import (
    Table, MetaData, Column, Integer, String, 
    create_engine, ForeignKey, Date, Time, Enum
)

engine = create_engine('postgresql:///premier_league')

metadata = MetaData()

competition_table = Table(
    'competition', metadata,
    Column('idcompetition', Integer, primary_key=True),
    Column('nomcompetition', String(155))
)

saison_table = Table(
    'saison', metadata,
    Column('id_saison', Integer, primary_key=True),
    Column('annee', Integer)
)

equipe_table = Table(
    'equipe', metadata,
    Column('idequipe', Integer, primary_key=True),
    Column('nomequipe', String(250)),
    Column('idcompetition', Integer, ForeignKey('competition.idcompetition')),
    Column('idsaison', Integer, ForeignKey('saison.id_saison'))
)

joueur_table = Table(
    'joueur', metadata,
    Column('idjoueur', Integer, primary_key=True),
    Column('nomjoueur', String(250)),
    Column('position', String(100)),
    Column('nationalite', String(100)),
    Column('id_equipe', Integer, ForeignKey('equipe.idequipe'))
)

match_table = Table(
    'match', metadata,
    Column('idmatch', Integer, primary_key=True),
    Column('date_match', Date),
    Column('heure', Time),
    Column('round', String(100)),
    Column('venue', String(250)),
    Column('idteamhome', Integer, ForeignKey('equipe.idequipe')),
    Column('idteam_away', Integer, ForeignKey('equipe.idequipe')),
    Column('id_competition', Integer, ForeignKey('competition.idcompetition')),
    Column('id_saison', Integer, ForeignKey('saison.id_saison'))
)

resultatmatch_table = Table(
    'resultatmatch', metadata,
    Column('idresultat', Integer, primary_key=True),
    Column('idmatch', Integer, ForeignKey('match.idmatch')),
    Column('idequipe', Integer, ForeignKey('equipe.idequipe')),
    Column('butsmarques', Integer),
    Column('butsconcedes', Integer),
    Column('resultat', Enum('Victoire', 'Défaite', 'Nul', name='resultat_enum', create_type=True))
)

statistiquejoueur_table = Table(
    'statistiquejoueur', metadata,
    Column('idstats', Integer, primary_key=True),
    Column('idjoueur', Integer, ForeignKey('joueur.idjoueur')),
    Column('buts', Integer, default=0),
    Column('passesdecisives', Integer, default=0),
    Column('nbmatchesplayed', Integer, default=0),
    Column('cartonsjaunes', Integer, default=0),
    Column('cartonsrouges', Integer, default=0)
)

if __name__ == "__main__":
    try:
        metadata.create_all(engine)
        print("success")
    except Exception as e:
        print(f"error: {e}")
        

success


In [ ]:
# from sqlalchemy import insert, select
# from sqlalchemy import create_engine
# import pandas as pd


# engine = create_engine('postgresql:///premier_league')
# df_matchs = pd.read_csv('premier_league_matchs.csv')

# with engine.begin() as conn:
#     competitions = df_matchs['comp'].unique()
#     comp_dict = {}

#     for i, name in enumerate(competitions, start=1):
#         conn.execute(insert(competition_table).values(idcompetition=i, nomcompetition=name))
#         comp_dict[name] = i

#     conn.execute(insert(saison_table).values(id_saison=1, annee=2024))

#     # all_teams = sorted(set(df_matchs['team']).union(df_matchs['opponent']))
#     # premier_league_id = comp_dict.get('Premier League', 1)

#     # equipes_data = [
#     #     dict(idequipe=i, nomequipe=team, idcompetition=premier_league_id, idsaison=1)
#     #     for i, team in enumerate(all_teams, start=1)
#     # ]
#     # conn.execute(insert(equipe_table), equipes_data)

#     print(f"inserted: {len(competitions)} competitions, 1 saison, equipes")
#     for c in competitions:
#         print(f"  - {c}")



inserted: 7 competitions, 1 saison, equipes
  - Premier League
  - EFL Cup
  - Champions Lg
  - FA Cup
  - FA Community Shield
  - Conf Lg
  - Europa Lg


In [ ]:

# df_matchs_opp_comp = df_matchs[['opponent', 'comp']].drop_duplicates()
# df_matchs_team_comp = df_matchs[['team', 'comp']].drop_duplicates()
# df_matchs_opp_comp = df_matchs_opp_comp.rename(columns={'opponent': 'team'})
# all_team_comps = pd.concat([df_matchs_team_comp, df_matchs_opp_comp]).drop_duplicates()
# team_to_comp_map = all_team_comps.groupby('team')['comp'].first().to_dict()


# equipes_data = [] 
# all_teams = sorted(set(df_matchs['team']).union(df_matchs['opponent']))

# with engine.begin() as conn:
    

#     for idx, team_name in enumerate(all_teams):
#         comp_name = team_to_comp_map.get(team_name)
            
#         comp_id = None
#         if comp_name:
#             comp_id_query = select(competition_table.c.idcompetition).where(
#                 competition_table.c.nomcompetition == comp_name
#             )
#             comp_id = conn.execute(comp_id_query).scalar()
#         else : 
#             print(f'error: competition not found for team {team_name}')
        
        
#         equipes_data.append(
#             dict(idequipe=idx, nomequipe=team_name, idcompetition=comp_id, idsaison=1)
#         )
        
 
#     conn.execute(insert(equipe_table), equipes_data)

#     print("insertion complete.")

Insertion complete.


In [13]:
import pandas as pd
from sqlalchemy import create_engine, insert, select
from datetime import datetime

try:
    df_players = pd.read_csv('premier_league_stats_updated.csv')
    df_matchs = pd.read_csv('finale_data_mathcs.csv')
except FileNotFoundError:
    print("Error: CSV files not found. Make sure 'premier_league_stats_updated.csv' and 'finale_data_mathcs.csv' are in the same directory.")
    exit()

saison_id_map = {}
comp_id_map = {}
team_id_map = {}



with engine.begin() as conn:

    saison_annee = 2024
    saison_insert_stmt = insert(saison_table).values(
        annee=saison_annee
    ).returning(saison_table.c.id_saison)
    
    saison_id_2024 = conn.execute(saison_insert_stmt).scalar()
    saison_id_map[saison_annee] = saison_id_2024
    print(f"Inserted Saison '2024' with ID: {saison_id_2024}")

    competitions = df_matchs['comp'].unique()
    for comp_name in competitions:
        comp_insert_stmt = insert(competition_table).values(
            nomcompetition=comp_name
        ).returning(competition_table.c.idcompetition)
        
        comp_id = conn.execute(comp_insert_stmt).scalar()
        comp_id_map[comp_name] = comp_id
    print(f"Inserted {len(comp_id_map)} competitions. Map: {comp_id_map}")

    
    pl_comp_id = comp_id_map.get('Premier League')
    if pl_comp_id is None:
        pl_comp_id = list(comp_id_map.values())[0]

    all_teams = sorted(set(df_matchs['team']).union(df_matchs['opponent']))
    for team_name in all_teams:
        team_insert_stmt = insert(equipe_table).values(
            nomequipe=team_name,
            idcompetition=pl_comp_id,
            idsaison=saison_id_2024
        ).returning(equipe_table.c.idequipe)
        
        team_id = conn.execute(team_insert_stmt).scalar()
        team_id_map[team_name] = team_id
    print(f"Inserted {len(team_id_map)} teams.")

    
    print("Inserting players and player stats...")
    player_id_map = {}

    for _, r in df_players.iterrows():
        team_id = team_id_map.get(r['team'])
        if team_id is None:
            print(f"Warning: Team '{r['team']}' not found. Skipping player '{r['player']}'.")
            continue

        player_insert_stmt = insert(joueur_table).values(
            nomjoueur=r['player'],
            position=r['position'],
            nationalite=r['nationality'],
            id_equipe=team_id
        ).returning(joueur_table.c.idjoueur)
        
        player_id = conn.execute(player_insert_stmt).scalar()

        if r['player'] not in player_id_map:
             player_id_map[r['player']] = player_id

        stats_data = {
            'idjoueur': player_id,
            'buts': int(r['goals']),
            'passesdecisives': int(r['assists']),
            'nbmatchesplayed': int(r['games']),
            'cartonsjaunes': int(r['cards_yellow']),
            'cartonsrouges': int(r['cards_red'])
        }
        conn.execute(insert(statistiquejoueur_table).values(**stats_data))
        
    print(f"Inserted {len(df_players)} players and their stats records.")

    
    print("Inserting matches and match results...")
    result_map = {'W': 'Victoire', 'L': 'Défaite', 'D': 'Nul'}
    
    for _, r in df_matchs.iterrows():
        date = pd.to_datetime(r['date']).date()
        time_str = str(r['start_time']).split('(')[0].strip()
        try:
            time = datetime.strptime(time_str, '%H:%M').time()
        except ValueError:
            time = None

        if r['venue'] == 'Home':
            home_name, away_name = r['team'], r['opponent']
        else:
            home_name, away_name = r['opponent'], r['team']

        home_id = team_id_map.get(home_name)
        away_id = team_id_map.get(away_name)
        comp_id = comp_id_map.get(r['comp'])
        team_id_for_row = team_id_map.get(r['team'])

        if not all([home_id, away_id, comp_id, team_id_for_row]):
            print(f"Warning: Skipping match on {date} due to missing FKs.")
            continue

        match_insert_stmt = insert(match_table).values(
            date_match=date,
            heure=time,
            round=r['round'],
            venue=r['venue'],
            idteamhome=home_id,
            idteam_away=away_id,
            id_competition=comp_id,
            id_saison=saison_id_2024
        ).returning(match_table.c.idmatch)
        
        match_id = conn.execute(match_insert_stmt).scalar()

        resultat_str = result_map.get(r['result'])
        if resultat_str:
            resultat_data = {
                'idmatch': match_id,
                'idequipe': team_id_for_row,
                'butsmarques': int(str(r['goals_for']).split('(')[0].strip()),
                'butsconcedes': int(str(r['goals_against']).split('(')[0].strip()),
                'resultat': resultat_str
            }
            conn.execute(insert(resultatmatch_table).values(**resultat_data))
        else:
            print(f"Warning: No result ('W'/'L'/'D') found for match {match_id}. Skipping result entry.")

    print(f"Successfully inserted {len(df_matchs)} matches and their results.")

print("All data insertion complete.")

Inserted Saison '2024' with ID: 2
Inserted 7 competitions. Map: {'Premier League': 8, 'EFL Cup': 9, 'Champions Lg': 10, 'FA Cup': 11, 'FA Community Shield': 12, 'Conf Lg': 13, 'Europa Lg': 14}
Inserted 82 teams.
Inserting players and player stats...
Inserted 702 players and their stats records.
Inserting matches and match results...
Successfully inserted 488 matches and their results.
All data insertion complete.


In [9]:
from sqlalchemy import text
from sqlalchemy import create_engine

engine = create_engine('postgresql:///premier_league')

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE statistiquejoueur, match, joueur, equipe, saison, competition RESTART IDENTITY CASCADE"))


In [14]:
from sqlalchemy import text, create_engine

engine = create_engine('postgresql:///premier_league')

with engine.connect() as conn:
    tables = ['competition', 'saison', 'equipe', 'joueur', 'match', 'statistiquejoueur']
    for table in tables:
        print(f"\n--- {table} ---")
        result = conn.execute(text(f"SELECT * FROM {table}"))
        for row in result:
            print(row)



--- competition ---
(8, 'Premier League')
(9, 'EFL Cup')
(10, 'Champions Lg')
(11, 'FA Cup')
(12, 'FA Community Shield')
(13, 'Conf Lg')
(14, 'Europa Lg')

--- saison ---
(2, 2024)

--- equipe ---
(83, 'AFC Wimbledon', 8, 2)
(84, "Acc'ton Stanley", 8, 2)
(85, 'Arsenal', 8, 2)
(86, 'Aston Villa', 8, 2)
(87, 'Barnsley', 8, 2)
(88, 'Barrow', 8, 2)
(89, 'Bolton', 8, 2)
(90, 'Bournemouth', 8, 2)
(91, 'Brentford', 8, 2)
(92, 'Brighton', 8, 2)
(93, 'Bristol Rovers', 8, 2)
(94, 'Bromley', 8, 2)
(95, 'Burnley', 8, 2)
(96, 'Cardiff City', 8, 2)
(97, 'Chelsea', 8, 2)
(98, 'Crawley Town', 8, 2)
(99, 'Crystal Palace', 8, 2)
(100, 'Doncaster', 8, 2)
(101, 'Everton', 8, 2)
(102, 'Fulham', 8, 2)
(103, 'Ipswich Town', 8, 2)
(104, 'Leicester City', 8, 2)
(105, 'Leyton Orient', 8, 2)
(106, 'Liverpool', 8, 2)
(107, 'Luton Town', 8, 2)
(108, 'Manchester City', 8, 2)
(109, 'Manchester Utd', 8, 2)
(110, 'Millwall', 8, 2)
(111, 'Morecambe', 8, 2)
(112, 'Newcastle Utd', 8, 2)
(113, 'Norwich City', 8, 2)
(114,

In [15]:
from sqlalchemy import select, desc


top_buteurs_query = select(
    joueur_table.c.nomjoueur,
    statistiquejoueur_table.c.buts
).select_from(

    joueur_table.join(statistiquejoueur_table) 
).order_by(
    
    desc(statistiquejoueur_table.c.buts) 
).limit(10)


print("--- Top 10 des Meilleurs Buteurs ---")
with engine.connect() as conn:
    results = conn.execute(top_buteurs_query)
    for i, row in enumerate(results, 1):
        
        print(f"{i}. {row.nomjoueur}: {row.buts} buts")

--- Top 10 des Meilleurs Buteurs ---
1. Mohamed Salah: 29 buts
2. Alexander Isak: 23 buts
3. Erling Haaland: 22 buts
4. Chris Wood: 20 buts
5. Bryan Mbeumo: 20 buts
6. Yoane Wissa: 19 buts
7. Ollie Watkins: 16 buts
8. Cole Palmer: 15 buts
9. Matheus Cunha: 15 buts
10. Jean-Philippe Mateta: 14 buts


In [ ]:
Joueurs_plus_decisifs = select(joueur_table.c.nomjoueur, 
                            statistiquejoueur_table.c.buts,
                            statistiquejoueur_table.c.passesdecisives
                               ).select_from(joueur_table.join(statistiquejoueur_table)).order_by(
                                   desc(statistiquejoueur_table.c.buts + statistiquejoueur_table.c.passesdecisives)
                               ).limit(10)
with engine.connect() as conn:
    results = conn.execute(Joueurs_plus_decisifs)
    for i, row in enumerate(results, 1):
        
        print(f"les joueurs les plus sont : {row.nomjoueur}")

les joueurs les plus sont : Mohamed Salah
les joueurs les plus sont : Alexander Isak
les joueurs les plus sont : Bryan Mbeumo
les joueurs les plus sont : Erling Haaland
les joueurs les plus sont : Ollie Watkins
les joueurs les plus sont : Yoane Wissa
les joueurs les plus sont : Cole Palmer
les joueurs les plus sont : Chris Wood
les joueurs les plus sont : Jarrod Bowen
les joueurs les plus sont : Matheus Cunha
